# 10 • Projet final : un prototype défendable

`[MÉTA | Formation 4-024 | Niveau Application | TP 10 | Mode CPU local]`

**Objectif :** Mener un parcours jusqu’à une recommandation argumentée.

**Temps indicatif :** Bloc final 180 min, avec évaluations. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP précédents.

**Preuves de réussite :** Notebook, comparaison, journal, analyse d’erreurs, fiche modèle et slide exécutive.

**Sources :** R00 et références du parcours choisi.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [1]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


Moteur disponible : 2.10.0+cpu | Données : /mnt/data/deep_learning_4_024/03_Travaux_pratiques/donnees


## 1. Choisir un seul parcours
**priorisation** : dossiers synthétiques, baseline logistique, MLP, capacité 30.

**vision** : chiffres publics, baseline linéaire, CNN, exactitude et F1 macro.

**sequence** : série synthétique, persistance, LSTM, MAE.

**documents** : textes artificiels, séparation par gabarit, TF-IDF appris sur train, logistique et MLP. Ce n’est pas un LLM.

Le choix de parcours ne change pas la rigueur du protocole. Les quatre chemins de correction peuvent être testés séparément ; une équipe n’a pas à les réaliser tous.

In [2]:
PARCOURS='priorisation'
OUVRIR_TEST=False
assert PARCOURS in ['priorisation','vision','sequence','documents']
fiche={'finalite':'À préciser par l’équipe','instant_prediction':'À préciser','utilisateur':'Agent en revue humaine, scénario fictif','action_autorisee':'Aucune décision réelle, prototype pédagogique','metrique_principale':'À justifier','limites_connues':['données non représentatives du métier réel']}
print(json.dumps(fiche,ensure_ascii=False,indent=2))

{
  "finalite": "À préciser par l’équipe",
  "instant_prediction": "À préciser",
  "utilisateur": "Agent en revue humaine, scénario fictif",
  "action_autorisee": "Aucune décision réelle, prototype pédagogique",
  "metrique_principale": "À justifier",
  "limites_connues": [
    "données non représentatives du métier réel"
  ]
}


## 2. Examiner les données et le code
Ouvrir modules/atelier.py : les classes BinaryMLP, TinyCNN, TinyLSTM et TextMLP y sont lisibles. Vérifier les fonctions de split et la transformation. La fonction project est une solution de référence compacte ; le livrable de l’équipe doit expliquer ses propres décisions et peut reprendre des cellules détaillées des TP précédents.

In [3]:
import inspect
from atelier import project
print(inspect.getsource(project))

def project(track='priorisation',open_test=False):
    """Solution de référence des quatre parcours, adaptée au temps disponible.
    Les choix se font sur validation. open_test=True est réservé à l’audit final.
    """
    seed_all();report={'parcours':track,'test_ouvert':bool(open_test)}
    if track=='priorisation':
        d=split_tabular(True);tr=d['train'];va=d['validation'];model=BinaryMLP()
        h=fit_model(model,tr,va,epochs=14)
        base=LogisticRegression(max_iter=300).fit(*tr);score=predict(model,va[0]);report['validation']=binary_metrics(va[1],score)
        report['baseline']=binary_metrics(va[1],base.predict_proba(va[0])[:,1]);report['capacite']=topk_metrics(va[1],score,30)
        if open_test:report['test_final']=binary_metrics(d['test'][1],predict(model,d['test'][0]))
    elif track=='vision':
        d=digits_data();model=TinyCNN();h=fit_model(model,d['train'],d['validation'],task='multi',epochs=12)
        X,y=d['validation'];p=predict(model,X,'multi').argmax(

## 3. Exécuter une référence puis documenter une expérience
Ne pas ouvrir le test pour choisir les hyperparamètres. Chaque essai consigne : hypothèse, changement, validation, décision. Une équipe avancée peut remplacer l’architecture ; elle doit garder le contrat de données.

In [4]:
rapport,modele,historique=project(PARCOURS,open_test=OUVRIR_TEST)
print(json.dumps(rapport,ensure_ascii=False,indent=2))
plot_history(historique,'Projet '+PARCOURS,'10_'+PARCOURS+'.png')
journal=pd.DataFrame([{'essai':1,'hypothese':'Référence reproductible','changement':'Aucun','partition':'validation','decision':'Comparer à la baseline avant un second essai'}])
journal.to_csv(RESULTS/'10_journal.csv',index=False)
assert rapport['test_ouvert'] is False

{
  "parcours": "priorisation",
  "test_ouvert": false,
  "validation": {
    "seuil": 0.5,
    "exactitude": 0.988,
    "precision": 0.8666666666666667,
    "rappel": 0.65,
    "f1": 0.7428571428571429,
    "roc_auc": 0.943972602739726,
    "precision_moyenne_AP": 0.82734994985761,
    "brier": 0.00863554235547781,
    "alertes": 15
  },
  "baseline": {
    "seuil": 0.5,
    "exactitude": 0.9826666666666667,
    "precision": 1.0,
    "rappel": 0.35,
    "f1": 0.5185185185185185,
    "roc_auc": 0.89,
    "precision_moyenne_AP": 0.6364794325447037,
    "brier": 0.014992293791729198,
    "alertes": 7
  },
  "capacite": {
    "k": 30,
    "cas_pertinents": 17,
    "precision_a_k": 0.5666666666666667,
    "rappel_a_k": 0.85
  },
  "epoques": 14,
  "limite": "Jeu pédagogique public ou synthétique, aucune validation métier réelle."
}


## 4. Analyse d’erreurs obligatoire
Conserver au moins cinq erreurs ou cas difficiles. Les qualifier : signal ambigu, donnée dégradée, classe proche, score trop confiant, contexte absent. Pour les séquences, sélectionner des écarts importants et les replacer sur la frise. Pour les documents, discuter la similarité artificielle des gabarits.

**Vérification avant ouverture du test :** pipeline figé ; métrique et seuil figés ; journal exporté ; absence de variable future ; baseline présente. L’ouverture du test se fait après validation du formateur et ne sert pas à relancer la recherche. La dernière assertion de la cellule précédente doit être adaptée seulement lors de cette ouverture finale explicite.

## 5. Livrables
Un notebook propre ; une fiche modèle ; une slide exécutive ; une recommandation « poursuivre l’expérimentation », « conserver la baseline » ou « arrêter ce prototype », motivée par les preuves.

**Slide exécutive :** problème et données à gauche, pipeline et graphique au centre, métrique, risque et décision à droite. Aucun résultat d’entreprise ne doit être inventé.

**Barème /100 :** problème 15 ; données et split 15 ; baseline 10 ; architecture 10 ; expériences 15 ; métriques 15 ; erreurs 10 ; risques et décision 10. Le score brut ne rapporte aucun point sans protocole.

**Soutenance :** 3 minutes de présentation et 2 minutes de questions. Dans les 20 minutes prévues, quatre équipes maximum ; au-delà, adapter le format en galerie commentée ou évaluations simultanées sans augmenter les 18 heures.

**Extensions hors socle :** validation glissante, calibration indépendante, courbes de ressources, adaptation d’un grand modèle préentraîné autorisé.